# Generador de Señal PWM con Filtro Pasabajo
Simulación del comportamiento de una señal PWM (modulación por ancho de pulso) y su transformación en una señal analógica mediante un filtro digital pasabajo. El objetivo principal es visualizar cómo los parámetros del sistema afectan la forma de la señal y su contenido espectral, tanto en el dominio del tiempo como en el de la frecuencia.

### Objetivo
Analizar el efecto del ciclo de trabajo, la frecuencia de conmutación y los parámetros del filtro (orden y frecuencia de corte) sobre la señal PWM y su versión filtrada. Esto ayuda a entender cómo se produce una señal continua a partir de una secuencia digital de pulsos.

🔎 Explicación de la gráfica
✅ Dominio del tiempo
Se muestran dos señales:

Señal PWM: Pulsos binarios con ciclo de trabajo definido.

Señal filtrada: Respuesta del sistema tras pasar por un filtro pasabajo.

➝ Permite observar la transición de una señal digital a una forma de onda continua, útil en aplicaciones como DACs o controladores de motores.

✅ Dominio de la frecuencia
Se muestra el espectro de:

Señal PWM (línea punteada cian).

Señal filtrada (línea roja).

Respuesta del filtro (envolvente azul punteada).

➝ Facilita el análisis de armónicos y cómo el filtro elimina componentes de alta frecuencia.

🔧 Parámetros ajustables (interactivos)
Ciclo de trabajo (𝐷): Proporción del tiempo en alto de cada ciclo PWM.

Orden del filtro (𝑁): Controla la pendiente del filtro pasabajo.

Frecuencia de conmutación (𝐹𝑠𝑤): Velocidad a la que se generan los pulsos PWM.

Frecuencia de corte (𝐹𝑐): Límite superior que el filtro permite pasar.

Cantidad de ciclos: Cuántos ciclos PWM se simulan.

Respuesta transitoria: Permite mostrar la respuesta inicial del filtro o solo la parte estable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, freqz, lfilter
from ipywidgets import interact

def pwm_filter(D=1/3, N=1, Fsw=1000, Fc=200, Ciclos=5, Transitorio=0):
    plt.close('all')
    """
    Simula la generación de una señal PWM, su filtrado y análisis en frecuencia.
    Parámetros:
    D: Ciclo de trabajo (duty cycle)
    N: Orden del filtro
    Fsw: Frecuencia de conmutación de la señal PWM
    Fc: Frecuencia de corte del filtro pasabajas
    Ciclos: numero de ciclos a simular
    transitorio: 0 para estacionario, 1 para mostrar transitorio
    """
    #creacion del vector de tiempo
    puntos = int(1e4)
    t = np.arange(0, 1, 1 / puntos)

    #generacion de un ciclo pwm
    x = np.zeros(puntos)
    indice = round(D * puntos)
    x[:indice] = 1

    #repeticion de los 10 ciclos
    xx = np.tile(x, Ciclos * 10)

    #filtro pasabajas
    b, a = butter(N, Fc / (Fsw * puntos / 2), btype='low')

    #filtro
    yy = lfilter(b, a, xx)

    #transformada de fourier
    XX = np.fft.fftshift(abs(np.fft.fft(xx)) / len(xx))
    YY = np.fft.fftshift(abs(np.fft.fft(yy)) / len(yy))
    FF = (np.arange(len(xx)) / len(xx) - 0.5) * Fsw * puntos

    #generacion de graficas
    fig, axs = plt.subplots(2, 1, figsize=(12, 8))

    #grafica: respuesta en el tiempo
    indices = np.arange(Ciclos * puntos)
    axs[0].plot(indices / puntos / Fsw, xx[indices], 'b:', linewidth=1, label="Señal PWM")
    if Transitorio:
        axs[0].plot(indices / puntos / Fsw, yy[indices], 'r', linewidth=1, label="Señal Filtrada")
        axs[0].set_title(f"Respuesta Transitoria\nCiclo de Trabajo = {D:.2f}")
    else:
        axs[0].plot(indices / puntos / Fsw, yy[-len(indices):], 'r', linewidth=1, label="Señal Filtrada")
        axs[0].set_title(f"Respuesta Estacionaria\nCiclo de Trabajo = {D:.2f}")

    axs[0].set_xlabel("Tiempo [s]")
    axs[0].set_ylabel("Voltaje [V]")
    axs[0].grid(True, which='both', linestyle='--', linewidth=0.5)
    axs[0].legend()

    #grafica: dominio de la frecuencia
    axs[1].plot(FF, XX, 'c:', linewidth=1, label="Señal PWM")
    axs[1].plot(FF, YY, 'r', linewidth=1, label="Señal Filtrada")

    #envolvente del filtro
    w, h = freqz(b, a, worN=len(FF), fs=Fsw * puntos)
    axs[1].plot(w, abs(h) * max(YY), 'b-.',label="Envolvente del filtro")
    axs[1].plot(-w, abs(h) * max(YY), 'b-.')

    axs[1].set_xlim([-0.34 * (Fsw * 12), 0.34 * (Fsw * 12)])
    axs[1].set_xlabel("Frecuencia [Hz]")
    axs[1].set_ylabel("Peso de Armónico")
    axs[1].set_title("Espectro de Frecuencia")
    axs[1].grid(True, which='both', linestyle='--', linewidth=0.5)
    axs[1].legend()

    plt.tight_layout()
    plt.show()

# Uso de interact para manipular parámetros
def run_interactive():
    interact(
        pwm_filter,
        D=(0.1, 0.9, 0.05),
        N=(1, 5, 1),
        Fsw=(500, 5000, 500),
        Fc=(50, 1000, 50),
        Ciclos=(1, 10, 1),
        transitorio=(0, 1, 1),
    )

run_interactive()


interactive(children=(FloatSlider(value=0.3333333333333333, description='D', max=0.9, min=0.1, step=0.05), Int…